<a href="https://colab.research.google.com/github/Prajkta11222/cnn-architecture-comparison-cifar10/blob/main/LeNet_AlexNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

print("Data loaded! Training images:", len(trainset), "Test images:", len(testset))

100%|██████████| 170M/170M [35:38<00:00, 79.7kB/s]


Data loaded! Training images: 50000 Test images: 10000


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class LeNet(nn.Module):
    def __init__(self):
        super(LeNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16*5*5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

print("LeNet class ready!")

LeNet class ready!


In [ ]:
model = LeNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

import time
start_time = time.time()

for epoch in range(15):
    running_loss = 0.0
    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/15, Loss: {running_loss/len(trainloader):.4f}")

lenet_train_time = time.time() - start_time
print(f"LeNet Training Time: {lenet_train_time:.2f} seconds")

Epoch 1/15, Loss: 1.6530
Epoch 2/15, Loss: 1.3799
Epoch 3/15, Loss: 1.2440
Epoch 4/15, Loss: 1.1517
Epoch 5/15, Loss: 1.0793
Epoch 6/15, Loss: 1.0245
Epoch 7/15, Loss: 0.9781
Epoch 8/15, Loss: 0.9396
Epoch 9/15, Loss: 0.8990
Epoch 10/15, Loss: 0.8666
Epoch 11/15, Loss: 0.8369
Epoch 12/15, Loss: 0.8095
Epoch 13/15, Loss: 0.7802
Epoch 14/15, Loss: 0.7577
Epoch 15/15, Loss: 0.7360
LeNet Training Time: 201.05 seconds


In [ ]:
# Parameter count
lenet_params = sum(p.numel() for p in model.parameters())
print(f"LeNet Total Parameters: {lenet_params:,}")

# Test accuracy
correct = 0
total = 0
model.eval()
with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

lenet_accuracy = 100 * correct / total
print(f"LeNet Test Accuracy: {lenet_accuracy:.2f}%")

LeNet Total Parameters: 62,006
LeNet Test Accuracy: 64.62%


In [ ]:
class AlexNet(nn.Module):
    def __init__(self, num_classes=10):
        super(AlexNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256*4*4, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

print("AlexNet class ready!")

AlexNet class ready!


In [ ]:
model = AlexNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

start_time = time.time()

for epoch in range(15):
    running_loss = 0.0
    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/15, Loss: {running_loss/len(trainloader):.4f}")

alexnet_train_time = time.time() - start_time
print(f"AlexNet Training Time: {alexnet_train_time:.2f} seconds")

Epoch 1/15, Loss: 1.6597
Epoch 2/15, Loss: 1.2563
Epoch 3/15, Loss: 1.0861
Epoch 4/15, Loss: 0.9567
Epoch 5/15, Loss: 0.8756
Epoch 6/15, Loss: 0.7969
Epoch 7/15, Loss: 0.7368
Epoch 8/15, Loss: 0.6902
Epoch 9/15, Loss: 0.6395
Epoch 10/15, Loss: 0.6046
Epoch 11/15, Loss: 0.5631
Epoch 12/15, Loss: 0.5314
Epoch 13/15, Loss: 0.5021
Epoch 14/15, Loss: 0.4798
Epoch 15/15, Loss: 0.4552
AlexNet Training Time: 446.36 seconds


In [ ]:
alexnet_params = sum(p.numel() for p in model.parameters())
print(f"AlexNet Total Parameters: {alexnet_params:,}")

correct = 0
total = 0
model.eval()
with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

alexnet_accuracy = 100 * correct / total
print(f"AlexNet Test Accuracy: {alexnet_accuracy:.2f}%")

torch.save(model.state_dict(), 'alexnet_model.pth')
print("AlexNet model saved!")

AlexNet Total Parameters: 35,855,178
AlexNet Test Accuracy: 76.01%
AlexNet model saved!
